# 01 – EDA: dair-ai/emotion
Run with **Runtime → Run all** (from the top). Understand class balance, text length, duplicates/leakage and label noise **before** modelling. The text under each **Observation** heading is a DRAFT from our measured results: rewrite it in your own words and check each number against the output above it.

In [ ]:
# Setup: works in Colab after cloning the repo. Adjust the path if you cloned elsewhere.
import os
if os.path.exists('/content/ec417-miniproject-P24-623175-623179'):
    %cd /content/ec417-miniproject-P24-623175-623179/notebooks
import sys; sys.path.insert(0, '..')
os.makedirs('../results/figures', exist_ok=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter, defaultdict
from src.data import load_emotion, label_names, MODEL_NAME
ds = load_emotion(); names = label_names(ds); print(ds); print(names)

## 1. Class balance (per split)

In [ ]:
rows = {s: Counter(ds[s]['label']) for s in ds}
counts = pd.DataFrame(rows).T[list(range(6))]; counts.columns = names
print(counts)
pct = (counts.div(counts.sum(1), axis=0)*100).round(1)
print(pct)
pct.T.plot.bar(figsize=(7,3.5), ylabel='% of split'); plt.tight_layout(); plt.savefig('../results/figures/class_balance.png', dpi=150); plt.show()
print('majority-class accuracy on test:', counts.loc['test'].max()/counts.loc['test'].sum())

### Observation (DRAFT – rewrite)
Joy and sadness dominate; love and surprise are rare (surprise ≈3.6 % of training data). Predicting "joy" for every tweet gives ≈0.35 accuracy but only 0.086 macro-F1, so accuracy would hide failures on rare classes and macro-F1 is the right primary metric.

## 2. Text length → justify MAX_LENGTH = 64

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
lens = np.array([len(x) for x in tok(list(ds['train']['text']))['input_ids']])   # list(...) needed with recent `datasets` versions
print('mean %.1f | p95 %.0f | p99 %.0f | max %d' % (lens.mean(), np.percentile(lens,95), np.percentile(lens,99), lens.max()))
plt.figure(figsize=(6,3)); plt.hist(lens, bins=50); plt.axvline(64, color='r', ls='--'); plt.xlabel('tokens'); plt.tight_layout()
plt.savefig('../results/figures/token_lengths.png', dpi=150); plt.show()
print('fraction truncated at 64:', (lens>64).mean())

### Observation (DRAFT – rewrite)
Only 0.16 % of training tweets exceed 64 tokens, so truncating at 64 loses almost nothing and keeps training fast. (Add the mean / p95 / p99 / max printed above.)

## 3. Leakage check: duplicates inside and across splits

In [ ]:
tr, va, te = [set(ds[s]['text']) for s in ('train','validation','test')]
print('dups inside train:', len(ds['train']) - len(tr))
print('train∩val:', len(tr&va), ' train∩test:', len(tr&te), ' val∩test:', len(va&te))
d = defaultdict(set)
for t, l in zip(ds['train']['text'], ds['train']['label']): d[t].add(l)
print('texts with >1 label in train:', sum(len(v)>1 for v in d.values()))

In [ ]:
# Do the overlapping texts have the same label? (uses the first training label per text)
train_df = pd.DataFrame({'text': list(ds['train']['text']), 'label': list(ds['train']['label'])})
test_df  = pd.DataFrame({'text': list(ds['test']['text']),  'label': list(ds['test']['label'])})
m = test_df.merge(train_df.drop_duplicates('text'), on='text', suffixes=('_test','_train'))
print(len(m), 'test rows whose text also appears in train; same label:', int((m.label_test == m.label_train).sum()))
print(m.assign(test=m.label_test.map(dict(enumerate(names))), train=m.label_train.map(dict(enumerate(names))))[['text','test','train']].to_string())

import os
path = '../results/preds/distilbert_full_seed0.csv'
if os.path.exists(path):
    p = pd.read_csv(path); ov = p[p.text.isin(set(train_df.text))]
    print('main model (seed 0) accuracy on the overlapping test rows:', len(ov), round((ov.y_true == ov.y_pred).mean(), 3))

### Observation (DRAFT – rewrite)
Measured: 31 duplicated texts inside train; train∩val = 5, train∩test = 11, val∩test = 3; 30 training texts have more than one label. All 11 train∩test texts carry a *different* label in test than in train, and the main model (seed 0) got 1 of 11 right. The overlap therefore does not inflate scores; it adds ≈10 unavoidable errors (≈0.5 accuracy points). We keep the official split (as required) and report this as a limitation.

## 4. Sample tweets per class (label noise)

In [ ]:
for i, n in enumerate(names):
    print('\n==', n)
    for t in ds['train'].filter(lambda x: x['label']==i).shuffle(seed=0)[:4]['text']: print(' -', t)

### Observation (DRAFT – rewrite)
Several labels are ambiguous or arguably wrong, e.g. "i feel satisfied and sad at the same time" (joy) and "...bunion feeling extremely tender and painful" (love). Labels come from hashtags (distant supervision), which puts a ceiling on achievable accuracy and is consistent with the joy↔love and fear↔surprise confusions seen later.